<a href="https://colab.research.google.com/github/sophienala-ui/flight-delay-app/blob/main/Air_Quality_Prediction_SDG11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Air Quality Prediction Using Machine Learning
### SDG 11: Sustainable Cities and Communities

**Project objective:** Build a machine learning model that estimates the concentration of fine particulate matter (PM2.5) using other air-pollution measurements, weather conditions, time information, and monitoring-station location.

PM2.5 refers to tiny particles in the air. High concentrations can indicate poor air quality. Monitoring and estimating PM2.5 can help communities understand urban air pollution.

**Machine learning task:** Regression  
**Algorithm:** Random Forest Regressor  
**Target (y):** PM2.5 concentration  
**Main evaluation metrics:** MAE, RMSE, and R²

> This project estimates PM2.5 for an observation using measurements recorded at the same time. It is not a forecast of future air quality, and it does not calculate an official Air Quality Index (AQI).

## 1. Dataset

This project uses the **Beijing Multi-Site Air-Quality Data** from the UCI Machine Learning Repository.

- 420,768 hourly observations
- 12 air-quality monitoring stations in Beijing
- March 2013 to February 2017
- Includes pollutant measurements and weather variables
- Missing values are recorded as `NA`

The dataset is used here as a learning example. Its results describe the recorded Beijing data and should not automatically be applied to other cities.

**Dataset source:** [UCI Machine Learning Repository – Beijing Multi-Site Air Quality](https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data)

**Dataset citation:** Chen, S. (2017). *Beijing Multi-Site Air Quality* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5RK5G

## 2. Import libraries

We use:
- `pandas` and `numpy` to handle data.
- `matplotlib` and `seaborn` to create graphs.
- `scikit-learn` to train and evaluate the machine learning model.

In [ ]:
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 3. Load the dataset

The dataset is downloaded directly from UCI using its dataset ID (501). The code first uses the repository's original combined table. If that is unavailable, it combines the feature and target tables.

In [ ]:
air_quality = fetch_ucirepo(id=501)

# Prefer the original table because it contains all variables together.
if air_quality.data.original is not None:
    df = air_quality.data.original.copy()
else:
    df = pd.concat(
        [air_quality.data.features, air_quality.data.targets],
        axis=1
    )

print("Dataset shape:", df.shape)
display(df.head())

## 4. Understand the data

Important columns include:
- `PM2.5`: fine particulate matter concentration; this is what the model will estimate.
- `PM10`, `SO2`, `NO2`, `CO`, `O3`: other pollutant measurements.
- `TEMP`, `PRES`, `DEWP`, `RAIN`, `WSPM`: weather measurements.
- `year`, `month`, `day`, `hour`: date and time.
- `wd`: wind direction.
- `station`: monitoring station.

First, inspect the column names, data types, and missing values.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\\nData information:")
df.info()

print("\\nMissing values (top 20):")
display(df.isna().sum().sort_values(ascending=False).head(20))

print("\\nSummary statistics:")
display(df.describe(include="all").T.head(25))

## 5. Clean and prepare the data

We combine the year, month, day, and hour columns into one timestamp. Rows without a PM2.5 target are removed because the model cannot learn from rows where the answer is missing.

For predictor columns, missing numeric values are filled using the median, and missing categorical values are filled using the most frequent category. This keeps the preprocessing simple and avoids removing too many observations.

In [ ]:
df = df.copy()

# Standardize column names in case the source contains extra spaces.
df.columns = [str(c).strip() for c in df.columns]

# Convert the target and numeric measurements to numeric values.
numeric_candidates = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
    "year", "month", "day", "hour"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Create a timestamp for chronological splitting.
df["timestamp"] = pd.to_datetime(
    df[["year", "month", "day", "hour"]],
    errors="coerce"
)

# Keep rows with a known target and valid timestamp.
df = df.dropna(subset=["PM2.5", "timestamp"]).copy()
df = df.sort_values("timestamp").reset_index(drop=True)

print("Rows after removing missing target/timestamp:", len(df))
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())
display(df.head())

## 6. Explore air quality

The following graphs help us understand the data before training the model.

The first graph shows the distribution of PM2.5 measurements. The second graph shows average PM2.5 by month across the dataset.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["PM2.5"], bins=50, kde=True)
plt.title("Distribution of PM2.5 Concentrations")
plt.xlabel("PM2.5 concentration (dataset measurement units)")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.show()

monthly_avg = df.groupby("month")["PM2.5"].mean().reindex(range(1, 13))
plt.figure(figsize=(9, 5))
monthly_avg.plot(marker="o")
plt.title("Average PM2.5 by Month")
plt.xlabel("Month")
plt.ylabel("Average PM2.5 concentration")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

## 7. Select features and target

The model uses the following information to estimate PM2.5:
- Other pollutant measurements.
- Weather conditions.
- Month and hour.
- Monitoring station and wind direction.

The target column `PM2.5` is deliberately excluded from the input features. The timestamp is also excluded because its useful parts are already represented by year, month, day, and hour.

**Important:** Because other pollutant readings are used as inputs, the model estimates PM2.5 when those other measurements are available. It does not predict air quality hours or days in advance.

In [ ]:
target = "PM2.5"

numeric_features = [
    "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
    "year", "month", "day", "hour"
]
categorical_features = ["wd", "station"]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]
feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df[target].copy()

print("Features:", feature_columns)
print("Target:", target)
print("Number of observations:", len(X))

## 8. Split the data chronologically

Air-quality records are time-based. Therefore, we do not randomly shuffle the rows.

- The first 80% of observations (earlier dates) are used for training.
- The final 20% (later dates) are used for testing.

This checks how well the model performs on a later time period than the one it learned from.

In [ ]:
split_index = int(len(df) * 0.80)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("Training period:", df["timestamp"].iloc[0], "to", df["timestamp"].iloc[split_index - 1])
print("Testing period:", df["timestamp"].iloc[split_index], "to", df["timestamp"].iloc[-1])
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 9. Preprocess the features

Machine learning models need numerical inputs. We:
1. Fill missing numeric values with the training-set median.
2. Fill missing categorical values with the most frequent training-set category.
3. Convert categorical columns such as station and wind direction into numerical indicator columns.

The preprocessing is fitted only on the training data, then applied to the test data.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 10. Train the Random Forest model

A Random Forest combines many decision trees. Each tree makes its own estimate, and the model averages their predictions.

This can capture non-linear relationships between pollution, weather, and time. We limit the tree depth to help control training time and reduce overfitting.

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("Model training completed.")

## 11. Evaluate the model

We use three evaluation metrics:

- **MAE (Mean Absolute Error):** the average size of the prediction error. Lower is better.
- **RMSE (Root Mean Squared Error):** similar to MAE, but gives larger errors more weight. Lower is better.
- **R²:** measures how much variation in the target is explained by the model. A value closer to 1 generally indicates a better fit; it can also be negative.

In [ ]:
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

## 12. Compare actual and predicted values

This table compares a small sample of the model's predictions with the actual PM2.5 measurements in the test period.

In [ ]:
comparison = pd.DataFrame({
    "Actual PM2.5": y_test.iloc[:15].values,
    "Predicted PM2.5": predictions[:15]
})
comparison["Absolute Error"] = (
    comparison["Actual PM2.5"] - comparison["Predicted PM2.5"]
).abs()

display(comparison.round(2))

## 13. Visualize model performance

If the model predicts accurately, the points should generally be close to the diagonal line where predicted values equal actual values.

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, predictions, alpha=0.25)
low = min(y_test.min(), predictions.min())
high = max(y_test.max(), predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.title("Actual vs Predicted PM2.5")
plt.xlabel("Actual PM2.5")
plt.ylabel("Predicted PM2.5")
plt.tight_layout()
plt.show()

## 14. Feature importance

Random Forest can estimate how useful each input feature was when making predictions. Higher importance means the model relied more on that feature.

Feature importance shows how the model uses the data; it does not prove that a feature causes PM2.5 levels to change.

In [ ]:
fitted_preprocessor = model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = model.named_steps["regressor"].feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False).head(12)

display(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Top 12 Features Used by the Random Forest Model")
plt.xlabel("Feature importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 15. Compare with a simple baseline

A machine learning model should be compared with a basic method. Here, the baseline always predicts the median PM2.5 value from the training data.

If the Random Forest has lower MAE and RMSE than the baseline, it makes more useful estimates than simply predicting one typical value for every observation.

In [ ]:
baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

results = pd.DataFrame({
    "Model": ["Median baseline", "Random Forest"],
    "MAE": [baseline_mae, mae],
    "RMSE": [baseline_rmse, rmse],
    "R²": [r2_score(y_test, baseline_predictions), r2]
})

display(results.round(3))

## 16. Conclusion

This project developed a Random Forest Regression model to estimate PM2.5 concentrations using air-pollution measurements, weather conditions, time information, and monitoring-station location.

The model was evaluated on the later 20% of the dataset, while the earlier 80% was used for training. MAE, RMSE, and R² were used to evaluate its performance, and a median baseline was included for comparison.

**Complete this conclusion after running the notebook:**
- The Random Forest achieved an MAE of **[insert your MAE]**.
- Its RMSE was **[insert your RMSE]**.
- Its R² score was **[insert your R²]**.
- Compared with the baseline, the Random Forest **[performed better / did not perform better]**.

### Limitations
1. The data comes from Beijing, so the results may not represent other cities.
2. Some observations have missing measurements, which are filled during preprocessing.
3. The model estimates PM2.5 using measurements from the same observation time; it is not a future forecast.
4. Feature importance does not prove cause and effect.
5. The model does not calculate an official AQI category.

### Connection to SDG 11
This project supports **SDG 11: Sustainable Cities and Communities**, especially the goal of reducing the environmental impact of cities. Understanding air pollution through data can help communities and decision-makers monitor urban environmental conditions and consider ways to improve air quality.

## 17. References

- Chen, S. (2017). *Beijing Multi-Site Air Quality* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5RK5G
- United Nations. *Goal 11: Make cities and human settlements inclusive, safe, resilient and sustainable.* https://sdgs.un.org/goals/goal11

**Reminder:** Run every code cell from top to bottom in Google Colab before submitting. Use the actual metric values produced by your run; do not make up results.